# Insurance Claims Agent: ReAct and Feedback Loops

Companion notebook for **lesson P1** of the study notes. Lesson 4.4 triaged claims with a **prompt chain**: extract → assess → route, every step an LLM call, in a fixed order. Here the same job is done by a **ReAct agent**: the model decides which tool to call next, and your code runs the tools. Every error goes back to the model as an observation, so it can correct itself: a **feedback loop**. Lesson P2 does the same job as a workflow, and P3 compares all three.

1. **Tools**: check coverage, price the damage, apply the routing rules
2. **Final gate**: the rules the agent can't break
3. **Parser + executor**: read the `ACT:` line, run the tool, return `OBSERVE:`
4. **You play the model**: test the tools and gates by hand, no LLM needed
5. **The system prompt** and **the loop** (two exits: final answer or step limit)
6. **Run the agent** on six claims

**Run it in Colab, setup cell first.** It uses Colab's built-in models (no API key). The setup cell lists the available models; to choose one, uncomment its `MODEL = ...` line there.

In [ ]:
# Setup: Colab's built-in models, no API key needed. Run this cell first.
from google.colab import ai
print("Available models:", ", ".join(ai.list_models()))

MODEL = None                              # Colab's default; uncomment a line below to choose
# MODEL = "google/gemini-2.5-flash"
# MODEL = "google/gemini-2.5-pro"
# MODEL = "google/gemini-3.8-flash"

def get_completion(messages):
    """Send chat messages to the model. Colab's generate_text takes one prompt, so roles are inlined."""
    prompt = "\n\n".join(f"{m['role'].upper()}:\n{m['content']}" for m in messages) + "\n\nASSISTANT:\n"
    return ai.generate_text(prompt, model_name=MODEL) if MODEL else ai.generate_text(prompt)

## 1 · Tools

Plain Python functions. The model never runs them: it asks for one in an `ACT:` line, and the executor (section 3) runs it.

- `POLICIES` stands in for the insurer's policy system, like `get_calendar()` in lesson 4.2.
- `price_damage` gives the cost from a price table, so the model no longer guesses a number. In 4.4 the hail claim came out as Moderate $3,000 because the model guessed.
- `route_claim` holds 4.4's routing rules, in code.

In [1]:
import ast, json, re
from typing import Literal, Optional
from pydantic import BaseModel, Field, ValidationError

POLICIES = {   # stand-in for the policy system
    "C001": {"active": True,  "coverages": ["glass", "collision"],         "deductible": 0},
    "C002": {"active": True,  "coverages": ["collision"],                  "deductible": 500},
    "C003": {"active": True,  "coverages": ["collision", "comprehensive"], "deductible": 1000},
    "C004": {"active": True,  "coverages": ["comprehensive"],              "deductible": 500},
    "C005": {"active": True,  "coverages": ["collision"],                  "deductible": 500},
    "C006": {"active": False, "coverages": ["collision", "comprehensive"], "deductible": 500},
}
COVERAGE_FOR = {"glass": ["glass", "comprehensive"], "collision": ["collision"],
                "hail": ["comprehensive"], "vandalism": ["comprehensive"], "theft": ["comprehensive"]}

def check_coverage(claim_id: str, incident_type: str) -> dict:
    """Is the policy active, and does it cover this kind of incident?"""
    if claim_id not in POLICIES:
        raise ValueError(f"no policy found for claim {claim_id}")
    if incident_type not in COVERAGE_FOR:
        raise ValueError(f"incident_type must be one of: {', '.join(COVERAGE_FOR)}")
    policy = POLICIES[claim_id]
    covered = policy["active"] and any(c in policy["coverages"] for c in COVERAGE_FOR[incident_type])
    return {"policy_active": policy["active"], "covered": covered, "deductible": policy["deductible"]}

REPAIR_COST = {"windshield": 300, "glass": 250, "front": 3000, "rear": 1200, "side": 800,
               "roof": 1800, "hood": 1500, "door": 1100, "bumper": 900, "fender": 700,
               "quarter panel": 1400, "trunk": 1300, "mirror": 300, "light": 350, "engine": 6000}

def price_damage(areas: list) -> dict:
    """Typical repair cost per damaged area. An area damaged twice (two doors) is listed twice."""
    unknown = [a for a in areas if a not in REPAIR_COST]
    if not areas or unknown:
        raise ValueError(f"unknown areas {unknown}; use only: {', '.join(REPAIR_COST)}")
    return {"items": [[a, REPAIR_COST[a]] for a in areas], "total": sum(REPAIR_COST[a] for a in areas)}

COST_RANGES = {"Minor": (100, 1_000), "Moderate": (1_000, 5_000), "Major": (5_000, 50_000)}
GLASS_ONLY = {"windshield", "glass"}

def route_claim(severity: str, est_cost: float, areas: list) -> dict:
    """The routing rules from lesson 4.4. Refuses a cost that doesn't fit the severity."""
    if severity not in COST_RANGES:
        raise ValueError(f"severity must be one of: {', '.join(COST_RANGES)}")
    low, high = COST_RANGES[severity]
    if not low <= est_cost <= high:
        raise ValueError(f"{severity} damage must cost {low:,}-{high:,} USD, got {est_cost:,.0f}")
    if severity == "Major":
        return {"queue": "total_loss"}
    if severity == "Moderate":
        return {"queue": "material_damage"}
    return {"queue": "glass" if set(areas) <= GLASS_ONLY else "fast_track"}

TOOLS = {"check_coverage": check_coverage, "price_damage": price_damage, "route_claim": route_claim}

## 2 · The final gate: rules the agent can't break

The agent chooses the path. These checks run in code when it calls `final_answer`, and a failed check goes back to the model as an error to fix:

- coverage must have been checked; a claim that isn't covered goes to `needs_review`
- the cost must be the `price_damage` total, not a guess
- the queue must be what the routing rules give for that severity and cost

In [2]:
class TriageDecision(BaseModel):          # what final_answer must contain
    claim_id: str
    queue: Literal["glass", "fast_track", "material_damage", "total_loss", "needs_review"]
    severity: Optional[Literal["Minor", "Moderate", "Major"]] = None
    est_cost: Optional[float] = None
    reason: str = Field(..., min_length=10)

def check_final(decision: TriageDecision, facts: dict) -> None:
    """facts holds the last result of each tool the agent called."""
    if "check_coverage" not in facts:
        raise ValueError("call check_coverage before final_answer")
    if decision.queue == "needs_review":
        return                                             # handing over to a human is always allowed
    if not facts["check_coverage"]["covered"]:
        raise ValueError("the claim is not covered, so the queue must be needs_review")
    if "price_damage" not in facts:
        raise ValueError("call price_damage before final_answer")
    priced = facts["price_damage"]
    if decision.est_cost != priced["total"]:
        raise ValueError(f"est_cost must be the price_damage total ({priced['total']}), got {decision.est_cost}")
    areas = [a for a, _ in priced["items"]]
    expected = route_claim(decision.severity, decision.est_cost, areas)["queue"]   # raises if severity doesn't fit
    if decision.queue != expected:
        raise ValueError(f"the routing rules give {expected} for {decision.severity}, not {decision.queue}")

## 3 · Parser and executor

The model's reply is text. The parser finds the `ACT:` line and reads the call with Python's own parser (`ast`), so `tool(arg=value)` becomes a function name and a dict of arguments. The executor runs the tool and returns an `OBSERVE:` line.

Errors don't crash the loop: they come back as `OBSERVE: ERROR ...` so the model can fix its call, the same pattern as lesson 2.1.

In [3]:
def parse_action(reply: str):
    """'... ACT: price_damage(areas=["door"])' -> ("price_damage", {"areas": ["door"]})"""
    if "ACT:" not in reply:
        raise ValueError("no ACT: line. Reply with THINK: ... then ACT: tool(arg=value, ...)")
    after = reply.split("ACT:", 1)[1]
    call_text = re.split(r"\n\s*(?:THINK|OBSERVE|ACT):", after)[0].strip().strip("`").strip()
    try:
        call = ast.parse(call_text, mode="eval").body
        if not isinstance(call, ast.Call) or not isinstance(call.func, ast.Name) or call.args:
            raise SyntaxError
        return call.func.id, {kw.arg: ast.literal_eval(kw.value) for kw in call.keywords}
    except (SyntaxError, ValueError):
        raise ValueError(f"could not read {call_text!r}. Write one call with keyword arguments: tool(arg=value)")

def execute(reply: str, facts: dict):
    """Run what the model asked for. Returns (observation, decision); decision is set only when accepted."""
    try:
        name, args = parse_action(reply)
        if name == "final_answer":
            decision = TriageDecision(**args)
            check_final(decision, facts)
            return "OBSERVE: accepted", decision
        if name not in TOOLS:
            raise ValueError(f"unknown tool {name}. Tools: {', '.join(TOOLS)}, final_answer")
        result = TOOLS[name](**args)
        facts[name] = result                               # the final gate checks these
        return f"OBSERVE: {json.dumps(result)}", None
    except (ValueError, TypeError) as e:                   # pydantic's ValidationError is a ValueError
        return "OBSERVE: ERROR " + " ".join(str(e).split())[:300], None

## 4 · You play the model

Before any LLM is involved, type the `ACT:` lines yourself and watch the tools and gates respond. These are the same calls the agent will make.

**Claim C004 (hail).** In 4.4 the model said Moderate, $3,000. The gate now refuses that until the answer matches the price table and the rules:

In [4]:
def play(claim_moves):
    facts = {}
    for reply in claim_moves:
        observation, decision = execute(reply, facts)
        print(reply.split("ACT:")[-1].strip()[:130])
        print("   ", observation[:150], "\n")

play([
    'ACT: check_coverage(claim_id="C004", incident_type="hail")',
    'ACT: price_damage(areas=["hood", "roof", "trunk", "mirror", "mirror", "glass"])',
    'ACT: final_answer(claim_id="C004", queue="material_damage", severity="Moderate", est_cost=3000, reason="hail dents")',
    'ACT: final_answer(claim_id="C004", queue="material_damage", severity="Moderate", est_cost=5450, reason="hail dents")',
    'ACT: final_answer(claim_id="C004", queue="total_loss", severity="Major", est_cost=5450, reason="hail on several panels")',
])

check_coverage(claim_id="C004", incident_type="hail")
    OBSERVE: {"policy_active": true, "covered": true, "deductible": 500} 

price_damage(areas=["hood", "roof", "trunk", "mirror", "mirror", "glass"])
    OBSERVE: {"items": [["hood", 1500], ["roof", 1800], ["trunk", 1300], ["mirror", 300], ["mirror", 300], ["glass", 250]], "total": 5450} 

final_answer(claim_id="C004", queue="material_damage", severity="Moderate", est_cost=3000, reason="hail dents")
    OBSERVE: ERROR est_cost must be the price_damage total (5450), got 3000.0 

final_answer(claim_id="C004", queue="material_damage", severity="Moderate", est_cost=5450, reason="hail dents")
    OBSERVE: ERROR Moderate damage must cost 1,000-5,000 USD, got 5,450 

final_answer(claim_id="C004", queue="total_loss", severity="Major", est_cost=5450, reason="hail on several panels")
    OBSERVE: accepted 



**Claim C006 (lapsed policy)**, plus two broken replies: no `ACT:` line, and a tool that doesn't exist.

In [5]:
play([
    'I think this one is a fast track claim.',
    'ACT: estimate_cost(claim="C006")',
    'ACT: check_coverage(claim_id="C006", incident_type="collision")',
    'ACT: final_answer(claim_id="C006", queue="fast_track", severity="Minor", est_cost=900, reason="small dent")',
    'ACT: final_answer(claim_id="C006", queue="needs_review", reason="policy is not active")',
])

I think this one is a fast track claim.
    OBSERVE: ERROR no ACT: line. Reply with THINK: ... then ACT: tool(arg=value, ...) 

estimate_cost(claim="C006")
    OBSERVE: ERROR unknown tool estimate_cost. Tools: check_coverage, price_damage, route_claim, final_answer 

check_coverage(claim_id="C006", incident_type="collision")
    OBSERVE: {"policy_active": false, "covered": false, "deductible": 500} 

final_answer(claim_id="C006", queue="fast_track", severity="Minor", est_cost=900, reason="small dent")
    OBSERVE: ERROR the claim is not covered, so the queue must be needs_review 

final_answer(claim_id="C006", queue="needs_review", reason="policy is not active")
    OBSERVE: accepted 



## 5 · The system prompt

The four parts from lesson 2.1: role and goal, the THINK/ACT format, the tools, and an example. The tool descriptions are the model's only view of the tools, so they carry the allowed values and the rules.

In [6]:
SYSTEM_PROMPT = """
You are a claims triage agent for an auto insurer. Your goal: decide which queue a First Notice
of Loss (FNOL) goes to, using the tools below. Never guess what a tool can tell you.

Reply with exactly one THINK/ACT pair, then stop and wait:
THINK: your reasoning about what to do next
ACT: one tool call, with keyword arguments

The system answers with OBSERVE: <result>. If it starts with ERROR, fix your call and try again.

Tools:
1. check_coverage(claim_id: str, incident_type: str)
   incident_type: glass, collision, hail, vandalism or theft. A car hit while parked is a collision.
   Returns whether the policy is active and covers this incident.
2. price_damage(areas: list[str])
   Typical repair cost per damaged area, and the total. Areas: windshield, glass, front, rear, side,
   roof, hood, door, bumper, fender, quarter panel, trunk, mirror, light, engine.
   List an area once per damaged part, e.g. both doors: ["door", "door"].
3. route_claim(severity: str, est_cost: float, areas: list[str])
   Applies the routing rules and returns the queue. Use the price_damage total as est_cost.
   Severity: Minor ($100-1,000), Moderate ($1,000-5,000), Major ($5,000-50,000).
4. final_answer(claim_id: str, queue: str, severity: str, est_cost: float, reason: str)
   Ends the task. queue must be the one route_claim returned. If the claim is not covered,
   use queue="needs_review" and leave out severity and est_cost.

Example:
THINK: A rock chipped the windshield, so this is a glass incident. First, check coverage.
ACT: check_coverage(claim_id="C009", incident_type="glass")
""".strip()

## 6 · The loop: two exits

Each turn: the model replies (THINK + ACT), the executor runs the call, and the `OBSERVE:` line goes back into the history. The loop ends one of two ways:

1. **Final answer**: `final_answer` passed the final gate.
2. **Step limit**: after `max_steps` turns the claim goes to `needs_review`, so a stuck agent can't loop forever.

In [7]:
def run_agent(claim_id: str, fnol: str, max_steps: int = 8, verbose: bool = False):
    messages = [{"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": fnol}]
    facts = {}
    for step in range(1, max_steps + 1):
        reply = get_completion(messages).split("OBSERVE:")[0].strip()   # drop any OBSERVE the model invents
        observation, decision = execute(reply, facts)
        if verbose:
            print(f"--- step {step}\n{reply}\n{observation}\n")
        if decision:
            return decision, step                                       # exit 1: accepted final answer
        messages += [{"role": "assistant", "content": reply},
                     {"role": "user", "content": observation}]
    return TriageDecision(claim_id=claim_id, queue="needs_review",
                          reason=f"no accepted answer after {max_steps} steps"), max_steps   # exit 2

## 7 · Run the agent

The five claims from lesson 4.4, plus C006 (lapsed policy). First one claim with the full THINK / ACT / OBSERVE trace:

In [8]:
sample_fnols = {
    "C001": "Claim ID: C001\nCustomer: John Smith\nVehicle: 2018 Toyota Camry\nIncident: While driving on the highway, "
            "a rock hit my windshield and caused a small chip about the size of a quarter. No other damage was observed.",
    "C002": "Claim ID: C002\nCustomer: Sarah Johnson\nVehicle: 2020 Honda Civic\nIncident: I was parked at the grocery store "
            "and returned to find someone had hit my car and dented the rear bumper and taillight. The taillight is broken "
            "and the bumper has a large dent.",
    "C003": "Claim ID: C003\nCustomer: Michael Rodriguez\nVehicle: 2022 Ford F-150\nIncident: I was involved in a serious "
            "collision at an intersection. The front of my truck is severely damaged, including the hood, bumper, radiator, "
            "and engine compartment. The airbags deployed and the vehicle is not drivable.",
    "C004": "Claim ID: C004\nCustomer: Emma Williams\nVehicle: 2019 Subaru Outback\nIncident: My car was damaged in a "
            "hailstorm. There are multiple dents on the hood, roof, and trunk. The side mirrors were also damaged and one "
            "window has a small crack.",
    "C005": "Claim ID: C005\nCustomer: David Brown\nVehicle: 2021 Tesla Model 3\nIncident: Someone keyed my car in the "
            "parking lot. There are deep scratches along both doors on the driver's side.",
    "C006": "Claim ID: C006\nCustomer: Priya Patel\nVehicle: 2017 Mazda 3\nIncident: I backed into a pole in a parking "
            "garage. The rear bumper is cracked and the trunk lid is dented.",
}

In [ ]:
decision, steps = run_agent("C004", sample_fnols["C004"], verbose=True)
print(decision)

Now all six. `steps` is the number of LLM calls; the 4.4 workflow made 2 per claim (extract, assess) when nothing failed.

In [ ]:
for claim_id, text in sample_fnols.items():
    d, steps = run_agent(claim_id, text)
    detail = f"{d.severity or '':8} ${d.est_cost or 0:>8,.0f}"
    print(f"{claim_id}  {d.queue:16} {detail}  steps={steps}  {d.reason[:50]}")

## Your turn

1. Set `max_steps=2` and run C004 again. Which exit fires?
2. Delete the `price_damage` rule from `check_final` and run C004 a few times. Does the model still use the price total?
3. Add a `theft` claim (car stolen, nothing to price). What should the agent do, and which gate gets in its way?
4. Add a tool `request_photos(claim_id, reason)` for claims the agent can't price from the text.